In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score


In [16]:
df = pd.read_csv('./data/titanic_train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=[col for col in drop_cols if col in drop_cols])

In [4]:
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])

In [5]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
scaler = StandardScaler()

In [8]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
num_cols = [col for col in num_cols if col in X_train.columns]

In [9]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')

In [10]:
embarked_train = ohe.fit_transform(X_train[['Embarked']])
embarked_test = ohe.transform(X_test[['Embarked']])

In [11]:
ohe_cols = ohe.get_feature_names_out(['Embarked'])
df_embarked_train = pd.DataFrame(embarked_train, columns=ohe_cols, index=X_train.index)
df_embarked_test = pd.DataFrame(embarked_test, columns=ohe_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=['Embarked']), df_embarked_train], axis=1)
X_test = pd.concat([X_test.drop(columns=['Embarked']), df_embarked_test], axis=1)

In [12]:
age_median = X_train['Age'].median()

X_train['Age'] = X_train['Age'].fillna(age_median)
X_test['Age'] = X_test['Age'].fillna(age_median)

fare_median = X_train['Fare'].median()

X_train['Fare'] = X_train['Fare'].fillna(fare_median)
X_test['Fare'] = X_test['Fare'].fillna(fare_median)

In [13]:
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

X_test[num_cols] = scaler.transform(X_test[num_cols])

In [14]:
model = RandomForestClassifier(random_state=42, max_depth=5, n_estimators=100)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [15]:
print("--- Метрики на тестовой выборке ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nДетальный отчет:")
print(classification_report(y_test, y_pred))

--- Метрики на тестовой выборке ---
Accuracy: 0.8101
F1-Score: 0.7069
ROC AUC: 0.8434

Детальный отчет:
              precision    recall  f1-score   support

           0       0.79      0.95      0.86       110
           1       0.87      0.59      0.71        69

    accuracy                           0.81       179
   macro avg       0.83      0.77      0.78       179
weighted avg       0.82      0.81      0.80       179

